# AI reconnaissance and enhanced malware coding; defenses (shared secrets, MFA, verify output)

# Coding problem, 1.4.e: AI-assisted reconnaissance detector

**Skill:** Analyze Risk, Mitigate Risk · **Maps to:** 1.4.A, 1.4.B · **Language:** Python 3 (pytest)

## Problem

A SIEM (Security Information and Event Management) system has collected a list of server log strings from Meridian Bank's network during a suspicious time window. Your job is to implement the detection engine: scan the log lines, apply four rule-based detectors for AI-assisted reconnaissance patterns, and return both a boolean verdict and the sorted list of defense controls that the detected patterns trigger.

## Function contract

Implement `solve(events: list[str]) -> dict` in the code editor (see the Reference section below).

## Input

`events`, a list of strings. Each string is one server log line (plain text, any format). The list may be empty.

Example input:
```
[
    "WARN  Port scan detected: 1-65535 sequential from 203.0.113.42 in 4s",
    "WARN  Login attempts: 312 usernames tried in 8s from 203.0.113.42",
    "INFO  SSH login success user=alice src=192.168.1.10",
]
```

## Output

A `dict` with exactly two keys:

| Key | Type | Description |
|-----|------|-------------|
| `"recon_detected"` | `bool` | `True` if at least one event matched any detection rule |
| `"recommended_defenses"` | `list[str]` | Sorted, de-duplicated list of defense name strings triggered by matched rules. Empty list `[]` when `recon_detected` is `False`. |

Example output for the input above:
```python
{
    "recon_detected": True,
    "recommended_defenses": [
        "ip-reputation-blocking",
        "login-velocity-controls",
        "rate-limit-port-scans",
    ]
}
```

## Detection rules (implement all four)

Apply these rules to each event string using `re.search` (case-insensitive where noted):

**Rule 1, Port scan:**
- Pattern: `re.search(r'(?i)port scan', event)` matches
- Action: set `recon_detected = True`; add `"rate-limit-port-scans"` to defenses

**Rule 2, DNS burst:**
- Pattern: `re.search(r'(?i)dns burst', event)` matches
- Action: set `recon_detected = True`; add `"dns-query-rate-limiting"` to defenses

**Rule 3, Login burst:**
- Pattern: `re.search(r'(?i)login attempts', event)` matches AND at least one integer >= 100 appears in the event (use `re.findall(r'\b(\d+)\b', event)`)
- Action: set `recon_detected = True`; add `"login-velocity-controls"` to defenses

**Rule 4, Subdomain enumeration:**
- Pattern: `re.search(r'(?i)subdomain', event)` matches AND `re.search(r'(?i)quer', event)` matches
- Action: set `recon_detected = True`; add `"dns-query-rate-limiting"` to defenses (if not already present)

**Always:** if `recon_detected` is `True` after processing all events, add `"ip-reputation-blocking"` to defenses.

## Defense name strings (exact, spelling matters for tests)

```
"dns-query-rate-limiting"
"ip-reputation-blocking"
"login-velocity-controls"
"rate-limit-port-scans"
```

## Examples

```
solve([])
-> {"recon_detected": False, "recommended_defenses": []}

solve(["INFO SSH login success user=alice", "INFO Backup completed"])
-> {"recon_detected": False, "recommended_defenses": []}

solve(["WARN Port scan detected: 1-65535 from 203.0.113.42 in 4s"])
-> {"recon_detected": True, "recommended_defenses": ["ip-reputation-blocking", "rate-limit-port-scans"]}

solve(["WARN DNS burst: 847 subdomain queries from 203.0.113.42 in 2s"])
-> {"recon_detected": True, "recommended_defenses": ["dns-query-rate-limiting", "ip-reputation-blocking"]}

solve(["WARN Login attempts: 50 usernames tried"])
-> {"recon_detected": False, "recommended_defenses": []}
```

Note the login-burst threshold: 50 < 100, so Rule 3 does not fire.

## Constraints

- Deterministic, offline, standard library only (`re`, `typing`).
- Must pass all cases in the tests.

## How to run

```
cd problem
Your code passes when every test below shows a checkmark (some are hidden until you run them).
```

To test your starter against solution:
```
Your code passes when every test below shows a checkmark (some are hidden until you run them).
Your code passes when every test below shows a checkmark (some are hidden until you run them).
```

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or composite) and must pass the tests unchanged.

### Entry point

```python
def solve(events: list[str]) -> dict
```

### Input (`events`)

`list[str]`, a list of zero or more server log line strings. Each string is plain text; no guaranteed format. May be empty.

Example:
```python
[
    "[2024-03-15 02:14:22] WARN  Port scan detected: 1-65535 sequential from 203.0.113.42 in 4s",
    "[2024-03-15 02:14:23] INFO  DNS query: www.meridianbank.com A 192.0.2.1",
    "[2024-03-15 02:14:24] WARN  DNS burst: 847 subdomain queries from 203.0.113.42 in 2s",
]
```

### Output (`result`)

`dict` with exactly two keys:

```python
{
    "recon_detected": bool,           # True if any rule matched
    "recommended_defenses": list[str] # sorted, de-duplicated defense name strings
}
```

When `recon_detected` is `False`, `recommended_defenses` must be `[]`.

### Detection rules

| Rule | Trigger condition | Defense added |
|------|-------------------|---------------|
| Port scan | `re.search(r'(?i)port scan', event)` | `"rate-limit-port-scans"` |
| DNS burst | `re.search(r'(?i)dns burst', event)` | `"dns-query-rate-limiting"` |
| Login burst | `re.search(r'(?i)login attempts', event)` AND any integer >= 100 in event | `"login-velocity-controls"` |
| Subdomain enum | `re.search(r'(?i)subdomain', event)` AND `re.search(r'(?i)quer', event)` | `"dns-query-rate-limiting"` |

Additionally: if `recon_detected` is `True` after all events are processed, add `"ip-reputation-blocking"`.

### Defense name strings (exact spelling)

```
"dns-query-rate-limiting"
"ip-reputation-blocking"
"login-velocity-controls"
"rate-limit-port-scans"
```

### Guarantees / rules

- Pure function, deterministic, no I/O, no network.
- Standard library only (`re`, `typing`).
- `recommended_defenses` must be sorted (`sorted(...)`) and de-duplicated (use a `set` internally, convert to sorted list at return).
- The login burst rule fires only when the integer extracted from the event is >= 100. Integers below 100 do not trigger the rule even if "login attempts" appears.
- Both DNS burst and subdomain enum rules map to the same defense string; de-duplication ensures it appears only once.

### Acceptance

```
Your code passes when every test below shows a checkmark (some are hidden until you run them).
```

All tests must pass when `IMPL` is unset (defaults to the reference solution) or set to any conforming implementation.


In [ ]:
# Your implementation
"""Starter code for 1.4.e. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""
import re
from typing import Dict, List


def solve(events: List[str]) -> Dict[str, object]:
    """Detect AI-assisted reconnaissance patterns in a list of log line strings.

    Args:
        events: list of server log line strings (any format; may be empty).
    Returns:
        {"recon_detected": bool, "recommended_defenses": sorted, de-duplicated list[str]}.
        When nothing matches, recommended_defenses must be [].
    """
    # TODO: apply the four detection rules from the problem statement above / the Reference section below:
    #   1. port scan          -> "rate-limit-port-scans"
    #   2. dns burst          -> "dns-query-rate-limiting"
    #   3. login attempts + an integer >= 100 -> "login-velocity-controls"
    #   4. subdomain + query  -> "dns-query-rate-limiting"
    # Then, if any rule fired, add "ip-reputation-blocking".
    # Return recommended_defenses sorted and de-duplicated.
    raise NotImplementedError("implement solve() — see the problem statement above")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------
def test_empty_log():
    assert impl.solve([]) == {"recon_detected": False, "recommended_defenses": []}


def test_clean_traffic():
    out = impl.solve([
        "INFO  SSH login success user=alice src=192.168.1.10",
        "INFO  Backup completed in 42s",
    ])
    assert out == {"recon_detected": False, "recommended_defenses": []}


def test_port_scan():
    out = impl.solve(["WARN  Port scan detected: 1-65535 from 203.0.113.42 in 4s"])
    assert out == {
        "recon_detected": True,
        "recommended_defenses": ["ip-reputation-blocking", "rate-limit-port-scans"],
    }


def test_dns_burst():
    out = impl.solve(["WARN  DNS burst: 847 subdomain queries from 203.0.113.42 in 2s"])
    assert out == {
        "recon_detected": True,
        "recommended_defenses": ["dns-query-rate-limiting", "ip-reputation-blocking"],
    }


def test_login_burst_below_threshold_does_not_fire():
    out = impl.solve(["WARN  Login attempts: 50 usernames tried"])
    assert out == {"recon_detected": False, "recommended_defenses": []}


# ---- hidden / edge cases -------------------------------------------------
def test_login_burst_at_threshold_fires():
    # 100 is the inclusive boundary (>= 100).
    out = impl.solve(["WARN  Login attempts: 100 usernames tried in 9s"])
    assert out == {
        "recon_detected": True,
        "recommended_defenses": ["ip-reputation-blocking", "login-velocity-controls"],
    }


def test_subdomain_enum_requires_both_tokens():
    # "subdomain" alone (no query token) must not trigger rule 4.
    out = impl.solve(["INFO  Discovered subdomain api.meridianbank.com"])
    assert out == {"recon_detected": False, "recommended_defenses": []}
    # Both tokens present -> fires.
    out2 = impl.solve(["WARN  312 subdomain queries observed"])
    assert out2 == {
        "recon_detected": True,
        "recommended_defenses": ["dns-query-rate-limiting", "ip-reputation-blocking"],
    }


def test_case_insensitive():
    out = impl.solve(["warn  PORT SCAN from 203.0.113.42"])
    assert out == {
        "recon_detected": True,
        "recommended_defenses": ["ip-reputation-blocking", "rate-limit-port-scans"],
    }


def test_defense_deduplicated_across_rules():
    # DNS burst (rule 2) and subdomain enum (rule 4) both map to dns-query-rate-limiting;
    # it must appear exactly once.
    out = impl.solve([
        "WARN  DNS burst: 900 lookups from 203.0.113.42",
        "WARN  847 subdomain queries from 203.0.113.42",
    ])
    assert out["recon_detected"] is True
    assert out["recommended_defenses"] == ["dns-query-rate-limiting", "ip-reputation-blocking"]
    assert out["recommended_defenses"].count("dns-query-rate-limiting") == 1


def test_full_multi_rule_scenario():
    out = impl.solve([
        "[02:14:22] WARN  Port scan detected: 1-65535 sequential from 203.0.113.42 in 4s",
        "[02:14:23] WARN  DNS burst: 847 subdomain queries from 203.0.113.42 in 2s",
        "[02:14:24] WARN  Login attempts: 312 usernames tried in 8s from 203.0.113.42",
        "[02:14:25] INFO  SSH login success user=alice src=192.168.1.10",
    ])
    assert out == {
        "recon_detected": True,
        "recommended_defenses": [
            "dns-query-rate-limiting",
            "ip-reputation-blocking",
            "login-velocity-controls",
            "rate-limit-port-scans",
        ],
    }


def test_defenses_sorted():
    out = impl.solve([
        "Port scan from x",
        "Login attempts: 500 from x",
    ])
    assert out["recommended_defenses"] == sorted(out["recommended_defenses"])


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
